## 環境設定

In [20]:
! pip install Backtesting

In [2]:
!wget http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
!tar -xzvf ta-lib-0.4.0-src.tar.gz
%cd ta-lib
!./configure --prefix=/usr
!make
!make install
!pip install Ta-Lib
import talib

--2024-01-03 02:56:34--  http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
Resolving prdownloads.sourceforge.net (prdownloads.sourceforge.net)... 204.68.111.105
Connecting to prdownloads.sourceforge.net (prdownloads.sourceforge.net)|204.68.111.105|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: http://downloads.sourceforge.net/project/ta-lib/ta-lib/0.4.0/ta-lib-0.4.0-src.tar.gz [following]
--2024-01-03 02:56:34--  http://downloads.sourceforge.net/project/ta-lib/ta-lib/0.4.0/ta-lib-0.4.0-src.tar.gz
Resolving downloads.sourceforge.net (downloads.sourceforge.net)... 204.68.111.105
Reusing existing connection to prdownloads.sourceforge.net:80.
HTTP request sent, awaiting response... 302 Found
Location: http://zenlayer.dl.sourceforge.net/project/ta-lib/ta-lib/0.4.0/ta-lib-0.4.0-src.tar.gz [following]
--2024-01-03 02:56:34--  http://zenlayer.dl.sourceforge.net/project/ta-lib/ta-lib/0.4.0/ta-lib-0.4.0-src.tar.gz
Resolving zenlayer.dl.s

# 修改套件問題


修改`/usr/local/lib/python3.10/dist-packages/backtesting/_plotting.py`的148行

改成
`new_bar_idx = new_index.get_indexer([mean_time], method='nearest')[0]`

In [21]:
import fileinput
import sys

# 文件路徑
file_path = '/usr/local/lib/python3.10/dist-packages/backtesting/_plotting.py'

# 需要修改的行號
line_to_edit = 148

# 新的內容
new_content = "                new_bar_idx = new_index.get_indexer([mean_time], method='nearest')[0]"

# 安全檢查：備份原始文件
backup_file_path = file_path + '.backup'
with open(file_path, 'r') as original_file, open(backup_file_path, 'w') as backup_file:
    backup_file.write(original_file.read())

# 嘗試修改文件
try:
    with fileinput.FileInput(file_path, inplace=True, backup='.bak') as file:
        for i, line in enumerate(file, start=1):
            if i == line_to_edit:
                print(new_content)  # 替換為新內容
            else:
                print(line, end='')  # 保留原始內容

    print("文件修改成功")
except Exception as e:
    print(f"發生錯誤: {e}", file=sys.stderr)
    # 發生錯誤時恢復原始文件
    with open(backup_file_path, 'r') as backup_file, open(file_path, 'w') as original_file:
        original_file.write(backup_file.read())
    print("文件已恢復到原始狀態。", file=sys.stderr)


文件修改成功


# 主程式

In [22]:
# 掛載自己的雲端硬碟
from google.colab import drive
drive.mount('/content/my-drive')

Drive already mounted at /content/my-drive; to attempt to forcibly remount, call drive.mount("/content/my-drive", force_remount=True).


In [44]:
from backtesting import Backtest, Strategy #引入回測和交易策略功能
from backtesting.lib import crossover #從lib子模組引入判斷交會功能
from backtesting.test import SMA,GOOG #從test子模組引入繪製均線功能
import pandas as pd #引入pandas讀取股價歷史資料CSV檔
from talib import abstract
import numpy as np

# 請更換成自己檔案在雲端上的路徑
data_path = "/content/my-drive/MyDrive/Colab_Notebooks_/programming_fin./EURUSD_M1_train.csv"

#匯入資料
df = pd.read_csv(data_path, names=GOOG.columns)
df.index = pd.to_datetime(df.index)
df.drop(index=df.index[0], inplace=True)
df = df.astype("float")

df.head()

<ipython-input-44-5b3d28a7df36>:12: DtypeWarning: Columns (1,2,3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, names=GOOG.columns)


,Open,High,Low,Close,Volume
2017-01-02 02:00:00,1.05155,1.05197,1.05155,1.05190,0.0
2017-01-02 02:01:00,1.05209,1.05209,1.05177,1.05179,0.0
2017-01-02 02:02:00,1.05177,1.05198,1.05177,1.05178,0.0
2017-01-02 02:03:00,1.05188,1.05200,1.05188,1.05200,0.0
2017-01-02 02:04:00,1.05196,1.05204,1.05196,1.05203,0.0


## 策略

請在這邊撰寫屬於自己的策略

如何下單請參考:
https://kernc.github.io/backtesting.py/doc/backtesting/backtesting.html#gsc.tab=0

以下範例示範簡單的黃金交叉策略，詳細說明可參考
https://rich01.com/what-is-golden-cross/

In [26]:
def RSV(arr: pd.Series, h: pd.Series, l: pd.Series, n: int) -> pd.Series:
  mxn = pd.Series(h).rolling(n).max()
  mn = pd.Series(l).rolling(n).min()

  return (arr[8]-mn)/(mxn-mn)*100

In [45]:
def _RSI(arr: np, n: int) -> pd.Series:

  return pd.Series(talib.RSI(arr,n))

In [125]:
class SmaCross(Strategy):
    # 定義兩個移動平均線的時間窗口長度
    n1 = 60
    n2 = 30
    n3 = 10
    k=0
    d=0
    def init(self):
        # 初始化策略
        close = self.data.Close
        high = self.data.High
        low = self.data.Low
        self.sma1 = self.I(SMA, close, self.n1)  #長線
        self.sma2 = self.I(SMA, close, self.n2)  #中線
        self.sma3 = self.I(SMA, close, self.n3)  #短線
        self.rsv = self.I(RSV, close, high, low, 14)
        self.k = self.k*(2/3)+self.rsv*(1/3)
        self.d = self.d*(2/3)+self.k*(1/3)
        self.rsi = self.I(_RSI, close, 14)

    def next(self):
        # 定義策略的執行邏輯
        if crossover(self.sma2,self.sma1) and crossover(self.sma3,self.sma2) and self.rsi>50 and self.rsi<85 :
            self.buy(size=10000000)  # 買入
        elif crossover(self.sma1,self.sma2) and crossover(self.sma2,self.sma3) and self.rsi<50 and self.rsi>15 :
            self.sell(size=10000000)  # 賣出



In [105]:
class KDCross(Strategy):
    # 定義兩個移動平均線的時間窗口長度
    n1 = 60
    n2 = 30
    n3 = 10
    k=0
    d=0
    def init(self):
        # 初始化策略
        close = self.data.Close
        high = self.data.High
        low = self.data.Low
        self.sma1 = self.I(SMA, close, self.n1)  #長線
        self.sma2 = self.I(SMA, close, self.n2)  #中線
        self.sma3 = self.I(SMA, close, self.n3)  #短線
        self.rsv = self.I(RSV, close, high, low, 14)
        self.k = self.k*(2/3)+self.rsv*(1/3)
        self.d = self.d*(2/3)+self.k*(1/3)
        self.rsi = self.I(_RSI, close, 14)

    def next(self):
        # 定義策略的執行邏輯
        if crossover(self.k,self.d) :
            self.buy(size=10000000,sl=self.data.Close[-1]*99/100)  # 買入
        elif crossover(self.d,self.k) :
            self.position.close()  # 賣出

## 執行回測

### 設定
* 初始金額：100000000
* 手續費：0.00005

In [126]:
bt = Backtest(df, SmaCross,cash=100000000, commission = 0.00005, exclusive_orders=True)
# 指定回測程式為bt，在Backtest函數中依序放入(資料來源、策略、現金、手續費)
output = bt.run()

output

<ipython-input-126-a0e23ba181de>:1: UserWarning: Data index is not sorted in ascending order. Sorting.
  bt = Backtest(df, SmaCross,cash=100000000, commission = 0.00005, exclusive_orders=True)


Start                     2017-01-02 02:00:00
End                       2022-12-30 16:58:00
Duration                   2188 days 14:58:00
Exposure Time [%]                   99.933676
Equity Final [$]                 103671233.24
Equity Peak [$]                  103837060.13
Return [%]                           3.671233
Buy & Hold Return [%]                1.772032
Return (Ann.) [%]                    0.486008
Volatility (Ann.) [%]                0.725268
Sharpe Ratio                         0.670108
Sortino Ratio                        1.007583
Calmar Ratio                         0.400257
Max. Drawdown [%]                   -1.214239
Avg. Drawdown [%]                   -0.015441
Max. Drawdown Duration      659 days 16:02:00
Avg. Drawdown Duration        1 days 17:10:00
# Trades                                  506
Win Rate [%]                        48.023715
Best Trade [%]                       4.437955
Worst Trade [%]                     -4.761103
Avg. Trade [%]                    

### 繪製分析圖

In [73]:
bt.plot()

/usr/local/lib/python3.10/dist-packages/backtesting/_plotting.py:122: UserWarning: Data contains too many candlesticks to plot; downsampling to '8H'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "
/usr/local/lib/python3.10/dist-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%d %b'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/usr/local/lib/python3.10/dist-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%m/%Y'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/usr/local/lib/python3.10/dist-packages/backtesting/_plotting.py:659: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
/usr/local/lib/python3.10/dist-packages/backtesting/_plotting.py:

GridPlot(id='p2257', ...)